# Pruebas de justificación con Gemma 4 12B IT

Se usa la segunda versión de las instrucciones para revisar si Gemma 4 12B IT explica la predicción de coautoría con evidencia de los artículos. Se prueban C001 y C002 en español e inglés.

Abre esta libreta junto a `casos.json`, `inventario_22_llm.csv` y `resultados.csv`. Con Ollama abierto y el modelo instalado, elige **Ejecutar todo**. Usa una libreta a la vez. Cada respuesta se guarda al terminar. Si interrumpes la prueba, vuelve a ejecutar la libreta para continuar. Si falta el CSV, la libreta lo crea. Si usas el formato anterior, conserva los intentos y actualiza sus columnas.

## 1. Leer los casos, el modelo y los intentos anteriores

In [10]:
import csv
import json
from copy import deepcopy
from pathlib import Path
from tempfile import NamedTemporaryFile
from time import perf_counter

import requests

id_modelo = "M05"
archivo = Path("resultados.csv")
columnas = [
    "id_caso", "id_modelo", "idioma", "intento", "solicitud_json", "estado",
    "tiempo_s", "respuesta", "error", "evaluacion", "motivo_evaluacion"
]

with open("casos.json", encoding="utf-8-sig") as f:
    datos = json.load(f)
casos = {caso["id_caso"]: caso for caso in datos["casos"]}

with open("inventario_22_llm.csv", encoding="utf-8-sig", newline="") as f:
    modelos = {fila["id_modelo"]: fila for fila in csv.DictReader(f)}
etiqueta = modelos[id_modelo]["etiqueta_ollama"]

# Si el CSV viene de las primeras pruebas, se conservan sus intentos.
encabezado = []
anteriores = []
if archivo.exists():
    with archivo.open(encoding="utf-8-sig", newline="") as f:
        lector = csv.reader(f)
        encabezado = next(lector, [])
        nombres = encabezado.copy()
        while nombres and nombres[-1] == "":
            nombres.pop()
        antiguas = columnas[:4] + ["fecha_utc", "version_ollama", "digest_modelo"] + columnas[4:]
        sin_idioma = [nombre for nombre in antiguas if nombre != "idioma"]
        if nombres and nombres not in (columnas, antiguas, sin_idioma):
            raise ValueError("Encabezado desconocido. Revisa resultados.csv antes de continuar.")
        for valores in lector:
            if not any(valores):
                continue
            if not nombres or any(valores[len(nombres):]):
                raise ValueError("Hay datos sin encabezado. Revisa resultados.csv antes de continuar.")
            fila = dict(zip(nombres, valores))
            anteriores.append({nombre: fila.get(nombre, "") for nombre in columnas})

# Se termina de escribir el nuevo formato antes de sustituir el archivo anterior.
if encabezado != columnas:
    temporal = NamedTemporaryFile(mode="w", encoding="utf-8-sig", newline="",
                                  dir=archivo.parent, delete=False)
    ruta_temporal = Path(temporal.name)
    try:
        with temporal:
            escritor = csv.DictWriter(temporal, fieldnames=columnas)
            escritor.writeheader()
            escritor.writerows(anteriores)
        ruta_temporal.replace(archivo)
    finally:
        ruta_temporal.unlink(missing_ok=True)

print("Modelo:", modelos[id_modelo]["nombre_modelo"], "|", etiqueta)

Modelo: Gemma 4 12B IT | gemma4:12b-it-q4_K_M


## 2. Preparar la solicitud

Cada modelo recibe la misma predicción y los textos originales de los artículos. El idioma elegido indica en qué lengua debe escribir la justificación. La referencia para evaluar la respuesta queda fuera de la solicitud.

In [11]:
instrucciones = """Genera una justificación semántica de la predicción de coautoría recibida.

1. Indica el resultado de la predicción de coautoría.
2. Identifica las coincidencias y diferencias entre las publicaciones de ambos autores.
3. Explica cómo se utiliza cada concepto o método compartido en los artículos.
4. Cita los identificadores de los artículos que respaldan cada afirmación.
5. Indica si la información disponible permite justificar semánticamente la predicción.

Consideraciones:
- No modifiques la predicción recibida.
- No atribuyas las coincidencias encontradas a las causas de la predicción.
- Distingue la similitud entre artículos de la probabilidad de coautoría.
- No califiques los valores de similitud sin un criterio establecido.
- La ausencia de palabras clave no significa que no existan coincidencias.
- Si faltan probabilidades o evidencia, indícalo.
- Utiliza únicamente la información proporcionada."""

def preparar_solicitud(id_caso, idioma):
    entrada = deepcopy(casos[id_caso]["entrada_llm"])
    for metodo in entrada["prediccion_coautoria"]["resultados_por_metodo"]:
        probabilidad = metodo.get("probabilidad_coautoria")
        umbral = metodo.get("umbral_aplicado")
        if umbral is None:
            umbral = datos["regla_coautoria"]["umbral_definitivo"]
        metodo["umbral_aplicado"] = umbral
        metodo["clasificacion_por_umbral"] = None
        # Solo se aplica el umbral a una probabilidad de coautoría.
        if probabilidad is not None and umbral is not None:
            if not (0 <= probabilidad <= 1 and 0 <= umbral <= 1):
                raise ValueError("La probabilidad y el umbral deben estar entre 0 y 1.")
            metodo["clasificacion_por_umbral"] = "favorable" if probabilidad >= umbral else "desfavorable"

    idioma_salida = datos["idiomas"]["instruccion_por_idioma"][idioma]
    return {
        "model": etiqueta,
        "messages": [
            {"role": "system", "content": instrucciones + "\n" + idioma_salida},
            {"role": "user", "content": json.dumps(entrada, ensure_ascii=False, indent=2)}
        ],
        "stream": False,
        "think": False,
        "options": {"temperature": 0, "num_ctx": 4096, "num_predict": 1024}
    }

## 3. Pedir la justificación a Ollama

Para cada solicitud se guarda el texto que devuelve el modelo, cuánto tarda y si termina de escribir. Esto permite revisar después la calidad de la respuesta.

In [12]:
url = "http://localhost:11434/api"

def pedir_justificacion(solicitud):
    inicio = perf_counter()
    registro = {"estado": "error", "respuesta": "", "error": ""}
    try:
        respuesta = requests.post(url + "/chat", json=solicitud, timeout=900)
        respuesta.raise_for_status()
        salida = respuesta.json()
        print("Motivo de término:", salida.get("done_reason"))
        print("Tokens generados:", salida.get("eval_count"))
        print("Tokens de entrada:", salida.get("prompt_eval_count"))
        print("Razonamiento recibido:", "Sí" if salida["message"].get("thinking") else "No")
        texto = salida["message"]["content"]
        registro["respuesta"] = texto
        if salida.get("done_reason") == "length":
            registro["estado"] = "truncada"
        elif salida.get("done") and texto.strip():
            registro["estado"] = "completa"
        else:
            registro["estado"] = "incompleta"
    except (requests.RequestException, ValueError, KeyError, TypeError) as error:
        registro["error"] = str(error)
    registro["tiempo_s"] = round(perf_counter() - inicio, 2)
    return registro

## 4. Ejecutar los dos casos en los dos idiomas

Se omiten las solicitudes idénticas que ya tienen una respuesta completa. Las solicitudes con error o con una respuesta truncada o incompleta se vuelven a intentar la próxima vez. Cada intento ocupa una fila. Las columnas `evaluacion` y `motivo_evaluacion` se llenan después de leer la respuesta y los artículos.

In [13]:
pendientes = []
for id_caso in ("C001", "C002"):
    for idioma in ("es", "en"):
        solicitud = preparar_solicitud(id_caso, idioma)
        completa = any(
            fila["id_modelo"] == id_modelo and fila["id_caso"] == id_caso
            and fila["idioma"] == idioma and fila["estado"] == "completa"
            and json.loads(fila["solicitud_json"]) == solicitud
            for fila in anteriores
        )
        if not completa:
            pendientes.append((id_caso, idioma, solicitud))

print("Pruebas pendientes:", len(pendientes), "de 4")
if pendientes:
    modelos_locales = requests.get(url + "/tags", timeout=10)
    modelos_locales.raise_for_status()
    instalados = {m["name"].lower() for m in modelos_locales.json()["models"]}
    if etiqueta.lower() not in instalados:
        raise ValueError("Instala el modelo en Ollama con: ollama pull " + etiqueta)

for id_caso, idioma, solicitud in pendientes:
    # La numeración continúa por caso y modelo, igual que en las primeras pruebas.
    intentos = [int(f["intento"]) for f in anteriores
                if f["id_modelo"] == id_modelo and f["id_caso"] == id_caso]
    intento = max(intentos, default=0) + 1
    print(id_caso, "|", idioma, "| intento", intento, flush=True)
    fila = pedir_justificacion(solicitud)
    fila.update({
        "id_caso": id_caso, "id_modelo": id_modelo, "idioma": idioma, "intento": intento,
        "solicitud_json": json.dumps(solicitud, ensure_ascii=False),
        "evaluacion": "", "motivo_evaluacion": ""
    })
    # Guardar únicamente las 11 columnas del archivo de resultados.
    fila = {nombre: fila.get(nombre, "") for nombre in columnas}
    with open(archivo, "a", encoding="utf-8", newline="") as f:
        csv.DictWriter(f, fieldnames=columnas).writerow(fila)
    anteriores.append(fila)
    print(fila["estado"], "|", fila["tiempo_s"], "s", fila["error"])
    print(fila["error"] if fila["error"] else fila["respuesta"])

Pruebas pendientes: 4 de 4
C001 | es | intento 5
Motivo de término: stop
Tokens generados: 443
Tokens de entrada: 1043
Razonamiento recibido: No
completa | 13.27 s 
A continuación se presenta la justificación semántica de la predicción de coautoría basada en la información proporcionada:

**1. Resultado de la predicción de coautoría**
La clasificación conjunta observada es "favorable", con el mensaje original indicando que "Es posible que los autores si colaboren en un futuro".

**2. Coincidencias y diferencias entre las publicaciones**
*   **Coincidencias:** Ambos autores comparten el uso del concepto "Gaussian Mixture Model" en sus trabajos.
*   **Diferencias:** Las aplicaciones prácticas y los dominios de estudio son distintos. El artículo P01 se centra en el procesamiento de imágenes de ultrasonido cerebral fetal (registro de nubes de puntos, máquinas de vectores de soporte), mientras que el artículo P02 se enfoca en el reconocimiento forense de voz (coeficientes cepstrales de frec